First best

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold, cross_val_score, cross_val_predict
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC, NuSVC
from sklearn.neural_network import MLPClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
import matplotlib.pyplot as plt

# 1. Chargement et préparation des données
df = pd.read_csv("parkinson.csv")
X = df.drop(columns=['ID', 'Recording', 'Status'])
y = df['Status']
X = pd.get_dummies(X, columns=['Gender'], drop_first=True)

# 2. Mise à l'échelle
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 3. Sélection de caractéristiques par méthode wrapper (First Best) avec Naïve Bayes
selector = SequentialFeatureSelector(GaussianNB(), direction='forward', cv=5)
X_selected = selector.fit_transform(X_scaled, y)
selected_features = X.columns[selector.get_support()]

# 4. Définition des classifieurs
models = {
    'Naïve Bayes': GaussianNB(),
    'c-SVM': SVC(kernel='rbf', C=1.0),
    'nu-SVM': NuSVC(nu=0.5, kernel='rbf'),
    'MLP': MLPClassifier(hidden_layer_sizes=(100,), max_iter=1000, solver='adam', random_state=42),
    'KNN': KNeighborsClassifier(n_neighbors=5),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42)
}

# 5. Évaluation par validation croisée 10-fold
results = []
kf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

for name, model in models.items():
    y_pred = cross_val_predict(model, X_selected, y, cv=kf)
    results.append({
        'Model': name,
        'Accuracy': accuracy_score(y, y_pred),
        'Precision': precision_score(y, y_pred),
        'Recall': recall_score(y, y_pred),
        'F1-Score': f1_score(y, y_pred)
    })

# 6. Création du DataFrame et affichage
results_df = pd.DataFrame(results)
display(results_df)

# 7. Courbe de performance
results_df.set_index("Model")[["Accuracy", "Precision", "Recall", "F1-Score"]].plot(kind="bar", figsize=(12, 6))
plt.title("Performance des Classifieurs avec Wrapper-based (First Best, base=Naïve Bayes)")
plt.ylabel("Score")
plt.ylim(0.6, 1)
plt.grid(axis='y')
plt.tight_layout()
plt.show()


 Greedy Stepwise

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC, NuSVC
from sklearn.neural_network import MLPClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
import matplotlib.pyplot as plt

# 1. Chargement et préparation
df = pd.read_csv("parkinson.csv")
X = df.drop(columns=['ID', 'Recording', 'Status'])
y = df['Status']
X = pd.get_dummies(X, columns=['Gender'], drop_first=True)

# 2. Mise à l’échelle
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 3. Sélection de caractéristiques avec Greedy Stepwise (forward + backward)
selector = SequentialFeatureSelector(
    estimator=GaussianNB(),
    direction='backward',  # Greedy Stepwise = backward + forward (tu peux tester 'forward' aussi)
    cv=5,
    n_jobs=-1
)
X_selected = selector.fit_transform(X_scaled, y)
selected_features = X.columns[selector.get_support()]

# 4. Définir les classifieurs
models = {
    'Naïve Bayes': GaussianNB(),
    'c-SVM': SVC(kernel='rbf', C=1.0),
    'nu-SVM': NuSVC(nu=0.5, kernel='rbf'),
    'MLP': MLPClassifier(hidden_layer_sizes=(100,), max_iter=1000, random_state=42),
    'KNN': KNeighborsClassifier(n_neighbors=5),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42)
}

# 5. Évaluation croisée 10-fold
results = []
kf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

for name, model in models.items():
    y_pred = cross_val_predict(model, X_selected, y, cv=kf)
    results.append({
        'Model': name,
        'Accuracy': accuracy_score(y, y_pred),
        'Precision': precision_score(y, y_pred),
        'Recall': recall_score(y, y_pred),
        'F1-Score': f1_score(y, y_pred)
    })

# 6. Affichage des résultats
results_df = pd.DataFrame(results)
print("\nRésultats avec Wrapper-based (Greedy Stepwise, base = Naïve Bayes):\n")
print(results_df)

# 7. Courbe de performance
results_df.set_index("Model")[["Accuracy", "Precision", "Recall", "F1-Score"]].plot(kind="bar", figsize=(12, 6))
plt.title("Performance des Classifieurs avec Greedy Stepwise (base = Naïve Bayes)")
plt.ylabel("Score")
plt.ylim(0.6, 1)
plt.grid(axis='y')
plt.tight_layout()
plt.show()


PSO Method

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold, cross_val_score, cross_val_predict
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC, NuSVC
from sklearn.neural_network import MLPClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
import pyswarms as ps
import matplotlib.pyplot as plt

# 1. Charger et préparer les données
df = pd.read_csv("parkinson.csv")
X = df.drop(columns=['ID', 'Recording', 'Status'])
y = df['Status']
X = pd.get_dummies(X, columns=['Gender'], drop_first=True)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 2. Définir la fonction d’évaluation du sous-ensemble (fitness)
def fitness_function(particles):
    scores = []
    for particle in particles:
        mask = particle > 0.5
        if np.sum(mask) == 0:
            scores.append(1.0)  # Mauvais score si aucune feature n'est sélectionnée
            continue
        X_selected = X_scaled[:, mask]
        score = cross_val_score(GaussianNB(), X_selected, y, cv=5, scoring='accuracy').mean()
        scores.append(1 - score)  # car on minimise
    return np.array(scores)

# 3. Configurer le PSO
n_particles = 20
dimensions = X_scaled.shape[1]

options = {'c1': 2, 'c2': 2, 'w': 0.9, 'k': 5, 'p': 2}
optimizer = ps.discrete.BinaryPSO(n_particles=n_particles, dimensions=dimensions, options=options)

# 4. Optimiser
best_cost, best_pos = optimizer.optimize(fitness_function, iters=30)

# 5. Extraire les meilleures features sélectionnées
selected_mask = best_pos > 0.5
X_selected_pso = X_scaled[:, selected_mask]
print(f"Nombre de caractéristiques sélectionnées : {np.sum(selected_mask)}")

# 6. Modèles à évaluer
models = {
    'Naïve Bayes': GaussianNB(),
    'c-SVM': SVC(kernel='rbf'),
    'nu-SVM': NuSVC(nu=0.5, kernel='rbf'),
    'MLP': MLPClassifier(hidden_layer_sizes=(100,), max_iter=2000, random_state=42),
    'KNN': KNeighborsClassifier(n_neighbors=5),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42)
}

# 7. Évaluation croisée
results = []
kf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

for name, model in models.items():
    y_pred = cross_val_predict(model, X_selected_pso, y, cv=kf)
    results.append({
        'Model': name,
        'Accuracy': accuracy_score(y, y_pred),
        'Precision': precision_score(y, y_pred),
        'Recall': recall_score(y, y_pred),
        'F1-Score': f1_score(y, y_pred)
    })

# 8. Résultats
results_df = pd.DataFrame(results)
print("\nRésultats avec Wrapper PSO (base = Naïve Bayes pour sélection des features)\n")
display(results_df)

# 9. Graphique
results_df.set_index("Model")[["Accuracy", "Precision", "Recall", "F1-Score"]].plot(kind="bar", figsize=(12, 6))
plt.title("Performance des Classifieurs avec Wrapper PSO (base = Naïve Bayes)")
plt.ylabel("Score")
plt.ylim(0.6, 1)
plt.grid(axis='y')
plt.tight_layout()
plt.show()
